In [1]:
from datasets import load_dataset
import ast
import warnings

warnings.filterwarnings("ignore")

# convert string to list
def convert_to_list(x):
    return ast.literal_eval(x)



ner = load_dataset("darrow-ai/LegalLensNER")

# load splits and convert tokens and ner_tags to list
train_ner = ner["train"]
train_ner = train_ner.map(lambda x: {'tokens': convert_to_list(x['tokens']), 'ner_tags': convert_to_list(x['ner_tags'])})
test_ner = ner["test"]
test_ner = test_ner.map(lambda x: {'tokens': convert_to_list(x['tokens']), 'ner_tags': convert_to_list(x['ner_tags'])})

print("Train-Test Split for NER: ", len(train_ner), len(test_ner))

print("Example Item:")
for key, value in train_ner[0].items():
    print(key, ":", value)

Train-Test Split for NER:  710 617
Example Item:
id : 006498c1-d337-4f67-bfe9-6f8c37c0b912
tokens : ['another', 'case', 'that', 'caught', 'my', 'attention', 'was', 'a', 'violation', 'of', 'employment', 'laws', 'by', 'a', 'popular', 'talent', 'agency', '.', 'they', 'were', 'found', 'guilty', 'of', 'requiring', 'their', 'non-union', 'employees', ',', 'including', 'the', 'plaintiff', ',', 'to', 'work', 'more', 'than', '31.33', 'hours', 'per', 'week', 'without', 'providing', 'overtime', 'compensation', '.', 'this', 'was', 'a', 'clear', 'violation', 'committed', 'on', 'their', 'non-union', 'employees', '.', 'its', 'disheartening', 'to', 'see', 'such', 'practices', 'in', 'the', 'entertainment', 'industry', '.']
ner_tags : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LAW', 'I-LAW', 'O', 'O', 'B-VIOLATED BY', 'I-VIOLATED BY', 'I-VIOLATED BY', 'O', 'O', 'O', 'O', 'O', 'O', 'B-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLAT

In [2]:
# check if token length and ner_tags length are equal
for i in range(len(train_ner)):
    if len(train_ner[i]['tokens']) != len(train_ner[i]['ner_tags']):
        print("Token and NER Tags Length Mismatch in Train Data")
        break

In [3]:
from datasets import ClassLabel, Sequence, Dataset, Features, Value, DatasetDict

# Step 1: Extract unique labels from LegalLensNER data
def extract_unique_labels(legal_lens_data):
    unique_labels = set()
    for entry in legal_lens_data:
        unique_labels.update(entry['ner_tags'])
    return sorted(unique_labels)

# Step 2: Assign integer labels similar to wikiann format
def assign_label_mapping(unique_labels):
    label_mapping = {'O': 0}  # Start with 'O' mapped to 0
    counter = 1
    for label in unique_labels:
        if label != 'O':
            label_mapping[label] = counter
            counter += 1
    return label_mapping

# Step 3: Apply the mapping to convert the dataset
def convert_to_wikiann_format(legal_lens_data, label_mapping):
    wikiann_data = []
    for entry in legal_lens_data:
        tokens = entry["tokens"]
        ner_tags = entry["ner_tags"]
        # Convert the ner_tags using the label_mapping
        wikiann_ner_tags = [label_mapping[tag] for tag in ner_tags]
        # Create the wikiann-formatted dictionary
        wikiann_entry = {
            "tokens": tokens,
            "ner_tags": wikiann_ner_tags
        }
        wikiann_data.append(wikiann_entry)
    return wikiann_data

# Convert list of dicts to dict of lists
def convert_list_to_dict(dataset_list):
    dict_data = {
        "tokens": [entry["tokens"] for entry in dataset_list],
        "ner_tags": [entry["ner_tags"] for entry in dataset_list]
    }
    return dict_data

# Example datasets
# Assume train_ner and test_ner are your original datasets
unique_labels = extract_unique_labels(train_ner)
label_mapping = assign_label_mapping(unique_labels)

# Convert to the desired format
train_data = convert_to_wikiann_format(train_ner, label_mapping)
test_data = convert_to_wikiann_format(test_ner, label_mapping)

# Convert the list of dicts to a dict of lists
train_data_dict = convert_list_to_dict(train_data)
test_data_dict = convert_list_to_dict(test_data)

# Step 1: Define the ClassLabel with the detected unique labels
class_label = ClassLabel(
    num_classes=len(unique_labels),
    names=unique_labels
)

# Step 2: Wrap the ClassLabel in a Sequence
ner_tags_feature = Sequence(
    feature=class_label,
    length=-1
)

# Step 3: Define the complete features for the dataset
features = Features({
    "tokens": Sequence(feature=Value("string"), length=-1),
    "ner_tags": ner_tags_feature
})

# Step 4: Create Dataset objects
train_dataset = Dataset.from_dict(train_data_dict, features=features)
test_dataset = Dataset.from_dict(test_data_dict, features=features)

# Convert to DatasetDict
raw_datasets = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

# Print the label mapping and a sample from the dataset
print("Label Mapping:", label_mapping)
print(raw_datasets["train"][0]["tokens"])
print(raw_datasets["train"][0]["ner_tags"])
print(raw_datasets["train"].features["ner_tags"])


Label Mapping: {'O': 0, 'B-LAW': 1, 'B-VIOLATED BY': 2, 'B-VIOLATED ON': 3, 'B-VIOLATION': 4, 'I-LAW': 5, 'I-VIOLATED BY': 6, 'I-VIOLATED ON': 7, 'I-VIOLATION': 8}
['another', 'case', 'that', 'caught', 'my', 'attention', 'was', 'a', 'violation', 'of', 'employment', 'laws', 'by', 'a', 'popular', 'talent', 'agency', '.', 'they', 'were', 'found', 'guilty', 'of', 'requiring', 'their', 'non-union', 'employees', ',', 'including', 'the', 'plaintiff', ',', 'to', 'work', 'more', 'than', '31.33', 'hours', 'per', 'week', 'without', 'providing', 'overtime', 'compensation', '.', 'this', 'was', 'a', 'clear', 'violation', 'committed', 'on', 'their', 'non-union', 'employees', '.', 'its', 'disheartening', 'to', 'see', 'such', 'practices', 'in', 'the', 'entertainment', 'industry', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 5, 0, 0, 2, 6, 6, 0, 0, 0, 0, 0, 0, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 3, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sequence(feature=Cl

In [4]:
# check if the tokens length and ner tags from raw_datasets match
for i in range(len(raw_datasets["train"])):
    if len(train_ner[i]['tokens']) != len(train_ner[i]['ner_tags']):
        print("Token and NER Tags Length Mismatch in Train Data")
        break

In [3]:
from transformers import AutoTokenizer

# adding prefix space for deberta model
model_name = "baptiste/deberta-finetuned-ner-connll-late-stop"
# model_name = "Jean-Baptiste/roberta-large-ner-english"
# download model and tokenizer
model_tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)

# checking fast tokenizer
model_tokenizer.is_fast

True

In [4]:
# tokenize pre-tokenized inputs
inputs = model_tokenizer(raw_datasets["train"][0]["tokens"], is_split_into_words=True)
inputs.tokens()

['<s>',
 'Ġanother',
 'Ġcase',
 'Ġthat',
 'Ġcaught',
 'Ġmy',
 'Ġattention',
 'Ġwas',
 'Ġa',
 'Ġviolation',
 'Ġof',
 'Ġemployment',
 'Ġlaws',
 'Ġby',
 'Ġa',
 'Ġpopular',
 'Ġtalent',
 'Ġagency',
 'Ġ.',
 'Ġthey',
 'Ġwere',
 'Ġfound',
 'Ġguilty',
 'Ġof',
 'Ġrequiring',
 'Ġtheir',
 'Ġnon',
 '-',
 'union',
 'Ġemployees',
 'Ġ,',
 'Ġincluding',
 'Ġthe',
 'Ġplaintiff',
 'Ġ,',
 'Ġto',
 'Ġwork',
 'Ġmore',
 'Ġthan',
 'Ġ31',
 '.',
 '33',
 'Ġhours',
 'Ġper',
 'Ġweek',
 'Ġwithout',
 'Ġproviding',
 'Ġovertime',
 'Ġcompensation',
 'Ġ.',
 'Ġthis',
 'Ġwas',
 'Ġa',
 'Ġclear',
 'Ġviolation',
 'Ġcommitted',
 'Ġon',
 'Ġtheir',
 'Ġnon',
 '-',
 'union',
 'Ġemployees',
 'Ġ.',
 'Ġits',
 'Ġdis',
 'heart',
 'ening',
 'Ġto',
 'Ġsee',
 'Ġsuch',
 'Ġpractices',
 'Ġin',
 'Ġthe',
 'Ġentertainment',
 'Ġindustry',
 'Ġ.',
 '</s>']

In [5]:
inputs.word_ids()

[None,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 25,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 36,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 53,
 53,
 54,
 55,
 56,
 57,
 57,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 None]

In [6]:
raw_datasets["train"][0]["ner_tags"]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 5,
 0,
 0,
 2,
 6,
 6,
 0,
 0,
 0,
 0,
 0,
 0,
 4,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 3,
 7,
 7,
 7,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [7]:
# function to align labels with tokens 
# --> special tokens: -100 label id (ignored by cross entropy),
# --> if tokens are inside a word, replace 'B-' with 'I-' 
def align_labels_with_tokens(labels, word_ids):
  aligned_label_ids = []
  previous_word_id = None
  for word_id in word_ids:
    if word_id is None:
      aligned_label_ids.append(-100)
    elif word_id != previous_word_id:
      # new word!
      label_id = labels[word_id]
      aligned_label_ids.append(label_id)
      previous_word_id = word_id
    else:
      # inside of word
      label = labels[previous_word_id]
      # if label starts with B- change it to I-
      # all B- label ids have an odd index in dataset features
      if label % 2 == 1:
        label += 1
      aligned_label_ids.append(label)

  return aligned_label_ids

# test on first sentence
test_labels = raw_datasets["train"][0]["ner_tags"]
test_word_ids = inputs.word_ids()
print(test_labels)
print(align_labels_with_tokens(test_labels, test_word_ids))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 5, 0, 0, 2, 6, 6, 0, 0, 0, 0, 0, 0, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 3, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 5, 0, 0, 2, 6, 6, 0, 0, 0, 0, 0, 0, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 3, 7, 7, 8, 8, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]


In [8]:
# define tokenize and align labels in one function to use on Dataset with map
def tokenize_and_align_labels(examples):
  tokenized_inputs = model_tokenizer(examples["tokens"], truncation=True,
                                     is_split_into_words=True)
  all_labels = examples["ner_tags"]
  new_labels = []
  for i, labels in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(labels, word_ids))

  tokenized_inputs["labels"] = new_labels
  return tokenized_inputs
# note: inputs are noter padded, will be done dynamically with data collator

In [9]:
# Now we can apply it on the whole dataset the optimized way with map
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

Map:   0%|          | 0/710 [00:00<?, ? examples/s]

Map:   0%|          | 0/617 [00:00<?, ? examples/s]

In [10]:
# using specific data collator to pad labels (specific to toekn classification task)
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer= model_tokenizer)
# test
batch = data_collator([tokenized_datasets["train"][i] for i in range(2)])
batch["labels"]

tensor([[-100,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,
            5,    0,    0,    2,    6,    6,    0,    0,    0,    0,    0,    0,
            4,    8,    8,    8,    8,    8,    8,    8,    8,    8,    8,    8,
            8,    8,    8,    8,    8,    8,    8,    8,    8,    8,    8,    8,
            8,    0,    0,    0,    0,    0,    0,    0,    3,    7,    7,    8,
            8,    7,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0, -100, -100, -100, -100, -100],
        [-100,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    4,
            8,    8,    8,    8,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0, 

In [11]:
# test original (no padding)
tokenized_datasets["train"][:2]["labels"]

[[-100,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  5,
  0,
  0,
  2,
  6,
  6,
  0,
  0,
  0,
  0,
  0,
  0,
  4,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  8,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  3,
  7,
  7,
  8,
  8,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  -100],
 [-100,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  4,
  8,
  8,
  8,
  8,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  -100]]

In [12]:
from datasets import load_metric

# Load the seqeval metric with trust_remote_code=True
metric = load_metric("seqeval", trust_remote_code=True)

ner_feature = raw_datasets["train"].features["ner_tags"]
# Assuming you have ner_feature and raw_datasets set up correctly
label_names = ner_feature.feature.names

# Example labels from your dataset
labels = raw_datasets["train"][0]["ner_tags"]

# Convert numerical labels to their string names
labels = [label_names[i] for i in labels]
print(labels)


['B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-VIOLATED BY', 'I-VIOLATED BY', 'B-LAW', 'B-LAW', 'B-VIOLATED ON', 'I-VIOLATED ON', 'I-VIOLATED ON', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'I-LAW', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW', 'B-LAW']


In [13]:
# test seqeval with manul predictions
predictions = labels.copy()
predictions[2] = "O"
metric.compute(predictions=[predictions], references=[labels])

{'LAW': {'precision': 1.0,
  'recall': 0.972972972972973,
  'f1': 0.9863013698630138,
  'number': 37},
 'VIOLATED BY': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'VIOLATED ON': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'VIOLATION': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'overall_precision': 1.0,
 'overall_recall': 0.975,
 'overall_f1': 0.9873417721518987,
 'overall_accuracy': 0.9850746268656716}

In [14]:
# Define metrics function with overall scores
import numpy as np
from tabulate import tabulate  # You can install this with `pip install tabulate`

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

    # Multiply by 100 and round to 2 decimal places
    precision = round(all_metrics["overall_precision"] * 100, 2)
    recall = round(all_metrics["overall_recall"] * 100, 2)
    f1 = round(all_metrics["overall_f1"] * 100, 2)
    accuracy = round(all_metrics["overall_accuracy"] * 100, 2)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }

In [15]:
### Define the model with labels
# define mappings from ID to labels and back
id2label = {str(i): label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [16]:
from transformers import AutoModelForTokenClassification

deberta_model = AutoModelForTokenClassification.from_pretrained(model_name,
                                                                id2label=id2label,
                                                                label2id=label2id,
                                                                ignore_mismatched_sizes=True)

# check
deberta_model.config.num_labels

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

9

In [17]:
## define training argument
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir = "tner-deberta-v3-large-conll2003",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=7,
    weight_decay=0.01,
    push_to_hub=False,
    load_best_model_at_end=True
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [18]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [19]:
from datasets import DatasetDict

# Split the train dataset into 90% train and 10% validation
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
print(train_val_split.keys())
# Rename the splits
train_val_split = DatasetDict({
    "train": train_val_split["train"],
    "val": train_val_split["test"],  # Rename 'test' to 'val'
})

# Check the number of samples
print(f"Training set size: {len(train_val_split['train'])}")
print(f"Validation set size: {len(train_val_split['val'])}")

dict_keys(['train', 'test'])
Training set size: 639
Validation set size: 71


In [20]:
from transformers import Trainer

trainer = Trainer(
    model=deberta_model,
    args=args,
    train_dataset=train_val_split["train"].shuffle(seed=42),
    eval_dataset=train_val_split["val"],  # Use the validation set as eval_dataset
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=model_tokenizer,
)

# Start training
trainer.train()

# After training is done, get the evaluation results
eval_results = trainer.evaluate()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/560 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

{'eval_loss': 0.23479107022285461, 'eval_precision': 93.65, 'eval_recall': 95.85, 'eval_f1': 94.73, 'eval_accuracy': 93.53, 'eval_runtime': 1.1183, 'eval_samples_per_second': 63.491, 'eval_steps_per_second': 8.048, 'epoch': 1.0}


  0%|          | 0/9 [00:00<?, ?it/s]

...

  0%|          | 0/9 [00:00<?, ?it/s]

{'eval_loss': 0.220835879445076, 'eval_precision': 96.68, 'eval_recall': 97.91, 'eval_f1': 97.29, 'eval_accuracy': 96.22, 'eval_runtime': 1.1545, 'eval_samples_per_second': 61.5, 'eval_steps_per_second': 7.796, 'epoch': 6.0}
{'loss': 0.1168, 'grad_norm': 2.7813711166381836, 'learning_rate': 2.1428571428571427e-06, 'epoch': 6.25}


  0%|          | 0/9 [00:00<?, ?it/s]

{'eval_loss': 0.2138214409351349, 'eval_precision': 96.82, 'eval_recall': 97.65, 'eval_f1': 97.23, 'eval_accuracy': 96.22, 'eval_runtime': 1.141, 'eval_samples_per_second': 62.229, 'eval_steps_per_second': 7.888, 'epoch': 7.0}
{'train_runtime': 264.8779, 'train_samples_per_second': 16.887, 'train_steps_per_second': 2.114, 'train_loss': 0.10597639488322394, 'epoch': 7.0}


  0%|          | 0/9 [00:00<?, ?it/s]

In [21]:
len(raw_datasets["test"])

617

In [22]:
# Load the best model (optional, Trainer does this automatically if args.load_best_model_at_end is True)
best_model = trainer.model

# Now we can apply it on the whole dataset the optimized way with map
tokenized_datasets_test = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["test"].column_names,
)

# Evaluate on the test dataset
test_results = trainer.evaluate(eval_dataset=tokenized_datasets_test)

print(test_results)

Map:   0%|          | 0/710 [00:00<?, ? examples/s]

Map:   0%|          | 0/617 [00:00<?, ? examples/s]

  0%|          | 0/89 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

{'eval_train_loss': 0.057259537279605865, 'eval_train_precision': 98.87, 'eval_train_recall': 98.11, 'eval_train_f1': 98.49, 'eval_train_accuracy': 98.21, 'eval_train_runtime': 10.6226, 'eval_train_samples_per_second': 66.839, 'eval_train_steps_per_second': 8.378, 'epoch': 7.0, 'eval_test_loss': 0.19431917369365692, 'eval_test_precision': 96.45, 'eval_test_recall': 95.53, 'eval_test_f1': 95.99, 'eval_test_accuracy': 94.34, 'eval_test_runtime': 9.7201, 'eval_test_samples_per_second': 63.477, 'eval_test_steps_per_second': 8.025}


In [54]:
# load xlsx file
import pandas as pd
df = pd.read_excel('testset_NER_LegalLens.xlsx')
df.head()

,id,tokens
0,14243437,"[""a"",""class"",""action"",""lawsuit"",""has"",""been"",""..."
1,18871551,"[""a"",""media"",""company"",""recently"",""came"",""unde..."
2,51514749,"[""a"",""national"",""bank"",""was"",""recently"",""held""..."
3,99676183,"[""a"",""recent"",""case"",""has"",""come"",""to"",""light""..."
4,14188948,"[""a"",""recent"",""incident"",""has"",""come"",""to"",""li..."


In [58]:
len(df)

380

In [57]:
import torch
import numpy as np

# Assuming `best_model` and `model_tokenizer` are already defined and loaded.

# List to store the predicted labels for each row in the DataFrame
predicted_label_names_per_row = []

# Process each row in the DataFrame
for i in range(len(df)):
    # Extract the tokens from the DataFrame
    tokens = df.loc[i, "tokens"]
    
    # Ensure tokens is a list of strings
    if isinstance(tokens, str):
        tokens = tokens.split()  # Convert a string of tokens to a list of tokens
    
    # Tokenize the input tokens
    inputs = model_tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding=True, truncation=True)
    
    # Move inputs to the same device as the model (if using GPU)
    inputs = {key: value.to(best_model.device) for key, value in inputs.items()}
    
    # Get the model outputs (logits)
    with torch.no_grad():
        outputs = best_model(**inputs)
    
    # Get the predicted label indices for the tokens in this row
    predicted_label_indices = outputs.logits.argmax(dim=-1).squeeze().cpu().numpy()
    
    # Convert the label indices to label names
    predicted_label_names = [label_names[label] for label in predicted_label_indices]
    
    # Append the predicted labels for this row to the list
    predicted_label_names_per_row.append(predicted_label_names)

# Add the predicted labels to the DataFrame as a new column
df["predicted_labels"] = predicted_label_names_per_row

# Display the updated DataFrame
df.head()

,id,tokens,predicted_labels
0,14243437,"[""a"",""class"",""action"",""lawsuit"",""has"",""been"",""...","[B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-L..."
1,18871551,"[""a"",""media"",""company"",""recently"",""came"",""unde...","[B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-L..."
2,51514749,"[""a"",""national"",""bank"",""was"",""recently"",""held""...","[B-LAW, B-LAW, B-VIOLATED ON, B-LAW, I-VIOLATE..."
3,99676183,"[""a"",""recent"",""case"",""has"",""come"",""to"",""light""...","[B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-L..."
4,14188948,"[""a"",""recent"",""incident"",""has"",""come"",""to"",""li...","[B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-LAW, B-L..."


In [62]:
# check if len of tokens and predicted labels are the same
len_tokens = df["tokens"].apply(lambda x: len(x.split() if isinstance(x, str) else x))
len_labels = df["predicted_labels"].apply(len)
len_mismatched = (len_tokens != len_labels).sum()
print(f"Number of rows with mismatched token and label lengths: {len_mismatched}")

Number of rows with mismatched token and label lengths: 380


In [59]:
# save to xlsx
df.to_excel('testset_NER_LegalLens_predicted.xlsx', index=False)

In [23]:
# Your label_mapping dictionary
label_mapping = {'O': 0, 'B-LAW': 1, 'B-VIOLATED BY': 2, 'B-VIOLATED ON': 3, 'B-VIOLATION': 4, 
                 'I-LAW': 5, 'I-VIOLATED BY': 6, 'I-VIOLATED ON': 7, 'I-VIOLATION': 8}

# Reverse the label mapping dictionary
reverse_label_mapping = {v: k for k, v in label_mapping.items()}

# Example sentence
sentence = " ".join(['someone', 'has', 'been', 'using', 'my', 'copyrighted', 'material', 'without', 'permission', '.', 'this', 'is', 'a', 'violation', 'of', 'my', 'intellectual', 'property', 'rights', 'under', 'the', 'digital', 'millennium', 'copyright', 'act', '.'])

# Tokenize the sentence
inputs = model_tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)

# Move inputs to the same device as the model
inputs = inputs.to(deberta_model.device)

# Perform the model prediction
with torch.no_grad():
    logits = deberta_model(**inputs).logits

# Get the predicted label indices
predicted_label_indices = logits.argmax(dim=-1)[0]

# Convert tokens back to words
tokens = model_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Convert label indices to label names using the reverse mapping
predicted_labels = [reverse_label_mapping[i.item()] for i in predicted_label_indices]

# Pair each token with its predicted label
result = list(zip(tokens, predicted_labels))

print(predicted_labels)

['I-VIOLATION', 'O', 'O', 'O', 'B-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'I-VIOLATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LAW', 'I-LAW', 'I-LAW', 'I-LAW', 'O', 'B-LAW', 'I-LAW', 'I-LAW', 'I-LAW', 'O', 'I-VIOLATION']


In [45]:
test_ner = ner["test"]
test_ner = test_ner.map(lambda x: {'tokens': convert_to_list(x['tokens']), 'ner_tags': convert_to_list(x['ner_tags'])})

test_tokens = [item['tokens'] for item in test_ner]
test_ner_tags = [item['ner_tags'] for item in test_ner]

In [49]:
len(test_tokens[0])

103

In [52]:
import torch

# Assuming `deberta_model` and `model_tokenizer` are already defined

# Your label mapping dictionary
label_mapping = {'O': 0, 'B-LAW': 1, 'B-VIOLATED BY': 2, 'B-VIOLATED ON': 3, 'B-VIOLATION': 4, 
                 'I-LAW': 5, 'I-VIOLATED BY': 6, 'I-VIOLATED ON': 7, 'I-VIOLATION': 8}

# Reverse the label mapping dictionary
reverse_label_mapping = {v: k for k, v in label_mapping.items()}

# Example test tokens from your dataset
test_tokens = [item['tokens'] for item in test_ner]  # Assuming `test_ner` is a list of dictionaries containing 'tokens'

# Initialize an empty list to store the output label lists
output_label_lists = []

# Iterate through the test tokens
for tokens in test_tokens:
    # Tokenize the sentence using the model tokenizer
    inputs = model_tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, padding=True)
    
    # Move inputs to the same device as the model
    inputs = inputs.to(deberta_model.device)
    
    # Perform the model prediction
    with torch.no_grad():
        logits = deberta_model(**inputs).logits
    
    # Get the predicted label indices
    predicted_label_indices = logits.argmax(dim=-1)
    
    # Convert the predicted label indices to label names using the reverse mapping
    predicted_labels = []
    for i, word_idx in enumerate(inputs.word_ids(batch_index=0)):
        if word_idx is not None:  # Skip special tokens like [CLS] and [SEP]
            label_idx = predicted_label_indices[0, i].item()
            predicted_labels.append(reverse_label_mapping[label_idx])
    
    # Add the predicted labels to the output label lists
    output_label_lists.append(predicted_labels)

# The `output_label_lists` now contains the predicted label lists for each input list in `test_tokens`

In [53]:
len(output_label_lists[0])

110